[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_03_main_transfer.ipynb)

# Module 6, Vision: Transfer Learning, Don't Train From Scratch If You Don't Have To

**Notebook:** `06_03_main_transfer`

## What we're doing

In [`06_02`](06_02_main_cnn.ipynb) we trained a CNN from random weights on 3,500 images. It worked, but the network had to discover *everything* from scratch, what edges look like, what a corner is, what a green texture looks like. On 3,500 images that's asking a lot.

**Transfer learning** changes the deal. We start from a network that has already learned a strong general-purpose visual vocabulary on a different (much larger) dataset, and we just retrain the last layer or two for our task. The intuition: the early layers of a vision network learn properties that are useful for *any* image (edges, colors, textures), so we may as well not relearn them.

We'll do this in two stages:

1. **Feature extraction.** Load a pretrained MobileNetV2 (trained on ImageNet, 1.4M images, 1000 classes), freeze all its weights, and train just a small classification head on top.
2. **Fine-tuning.** Unfreeze the top of the backbone and train it at a *much* lower learning rate, so the pretrained filters get nudged toward EuroSAT without being destroyed.

## The recipe

| Step | Tool | What it does |
| --- | --- | --- |
| Load | Zenodo + PIL | same EuroSAT 5,000-image sample, same 80/20 split as 06_01 / 06_02 |
| Resize | tf.image | 64x64 -> 96x96 (MobileNetV2 minimum) |
| Stage 1 | frozen backbone + dense head | feature extraction |
| Stage 2 | unfreeze top blocks + small lr | fine-tuning |
| Compare | accuracy table | head-to-head against 06_01 (RF) and 06_02 (CNN) |

**Comparison anchor.** Same 1,000-image test set as 06_01 and 06_02. Targets to beat: random forest at ~79%, CNN-from-scratch at ~85-88%. Transfer learning typically clears 95% on EuroSAT.

## 0) Setup

Same dependencies as 06_02. The pretrained MobileNetV2 weights download once on first use (~10 MB) and cache under `~/.keras/`.

In [ ]:
import os, zipfile

import numpy as np
import requests
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf

tf.keras.utils.set_random_seed(1955)
RNG = np.random.default_rng(1955)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 1) Load EuroSAT (matches 06_01 and 06_02 exactly)

Identical loader to the previous two notebooks. Same 5,000 sample, same `random_state=1955` split. The test set is the same 1,000 images, so test-accuracy numbers are directly comparable across all three notebooks.

In [ ]:
DATA_DIR = "assets/data"
ZIP_PATH = f"{DATA_DIR}/EuroSAT_RGB.zip"
EXTRACT_DIR = f"{DATA_DIR}/EuroSAT_RGB"
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    print("Downloading EuroSAT (~90 MB, one-time)...")
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

def find_class_dir(root):
    for r, dirs, _ in os.walk(root):
        if len(dirs) >= 8:
            return r
    raise RuntimeError(f"Could not find class folders under {root}")

CLASS_DIR = find_class_dir(EXTRACT_DIR)
label_names = sorted(os.listdir(CLASS_DIR))
num_classes = len(label_names)

N_SAMPLES = 5000
all_paths = []
for ci, cls in enumerate(label_names):
    for fn in os.listdir(os.path.join(CLASS_DIR, cls)):
        all_paths.append((os.path.join(CLASS_DIR, cls, fn), ci))
RNG.shuffle(all_paths)
sampled = all_paths[:N_SAMPLES]

X_img = np.zeros((N_SAMPLES, 64, 64, 3), dtype=np.uint8)
y     = np.zeros(N_SAMPLES, dtype=np.int64)
for i, (path, lab) in enumerate(sampled):
    X_img[i] = np.asarray(Image.open(path).convert("RGB"))
    y[i] = lab

X_trv, X_te, y_trv, y_te = train_test_split(
    X_img, y, test_size=0.20, stratify=y, random_state=1955
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_trv, y_trv, test_size=0.125, stratify=y_trv, random_state=1955
)
print(f"train: {X_tr.shape}    val: {X_val.shape}    test: {X_te.shape}")

## 2) Pipeline: resize to the backbone's expected size

MobileNetV2 was trained on ImageNet at 224x224. It will accept smaller inputs (down to 96x96) but won't go all the way down to our 64x64. We resize on the fly inside the `tf.data` pipeline.

**Important detail.** MobileNetV2 expects pixel values in `[-1, 1]`, not `[0, 1]`. Keras provides `mobilenet_v2.preprocess_input` which does the rescale, we use it inside the dataset map so the model always sees correctly-scaled inputs.

In [ ]:
BATCH_SIZE = 64
IMG_SIZE = (96, 96)
AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(
        lambda x, y: (preprocess(tf.image.resize(tf.cast(x, tf.float32), IMG_SIZE)), y),
        num_parallel_calls=AUTOTUNE,
    )
    if shuffle:
        ds = ds.shuffle(2048, seed=1955)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train = make_ds(X_tr, y_tr, shuffle=True)
ds_val   = make_ds(X_val, y_val)
ds_test  = make_ds(X_te, y_te)

for xb, yb in ds_train.take(1):
    print("X batch:", xb.shape, xb.dtype, " range:", float(xb.numpy().min()), float(xb.numpy().max()))

## 3) Stage 1, feature extraction (frozen backbone)

Build the model in four pieces:

1. **A flip layer** at the very top, same reasoning as 06_02. Active only at training time. Even with a strong pretrained backbone, flips help the head generalize on small data. (We skip `RandomRotation` for the same reason as 06_02, the boundary artifacts cost more than they gain at this image size.)
2. **Pretrained backbone**, MobileNetV2 with `include_top=False` (drop the original 1000-class ImageNet head), `weights="imagenet"`.
3. **`backbone.trainable = False`**, freeze every weight. The forward pass works as before; the backward pass touches nothing in the backbone. Effectively, the backbone becomes a fixed feature extractor.
4. **Small head**: global pool -> dropout -> 10-class softmax. This is the only part that learns in stage 1.

On a small dataset like ours, this stage usually does most of the work. The features ImageNet learned transfer remarkably well to satellite imagery, even though ImageNet has no satellite photos.

In [ ]:
backbone = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
backbone.trainable = False

tl_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SIZE + (3,)),
    # Flips only (same reasoning as 06_02). Active during training, identity at inference.
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    backbone,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
], name="mobilenetv2_frozen")

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable, total = (
    sum(int(np.prod(v.shape)) for v in tl_model.trainable_variables),
    tl_model.count_params(),
)
print(f"trainable params: {trainable:,} / {total:,}  ({100 * trainable / total:.2f}%)")

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    ),
]

history_stage1 = tl_model.fit(
    ds_train, validation_data=ds_val,
    epochs=40, callbacks=callbacks, verbose=2,
)

_, stage1_acc = tl_model.evaluate(ds_test, verbose=0)
print(f"\nStage 1 (frozen backbone) test accuracy: {stage1_acc:.3f}")

## 4) Stage 2, fine-tuning (unfreeze the top of the backbone)

Stage 1 leaves the backbone untouched, the pretrained features are useful but not *exactly* tuned to satellite imagery. Stage 2 fixes that by:

1. **Unfreezing** the top ~30 layers of the backbone (deeper, more task-specific features). The earliest layers (edge / color detectors) we leave frozen, they're already general-purpose.
2. **Compiling with a much lower learning rate** (`1e-5`, 100x smaller than stage 1). This is critical, large updates would wreck the pretrained weights.
3. **Re-fitting** for a few epochs.

The classic gotcha here is using the same learning rate as stage 1; the model overshoots and loses the pretrained knowledge. Small lr, few epochs.

In [ ]:
backbone.trainable = True
FREEZE_UPTO = len(backbone.layers) - 30   # unfreeze last 30 layers only
for layer in backbone.layers[:FREEZE_UPTO]:
    layer.trainable = False

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable = sum(int(np.prod(v.shape)) for v in tl_model.trainable_variables)
total     = tl_model.count_params()
print(f"trainable params after unfreeze: {trainable:,} / {total:,}  ({100 * trainable / total:.2f}%)")

history_stage2 = tl_model.fit(
    ds_train, validation_data=ds_val,
    epochs=25, callbacks=callbacks, verbose=2,
)

_, stage2_acc = tl_model.evaluate(ds_test, verbose=0)
print(f"\nStage 2 (fine-tuned) test accuracy: {stage2_acc:.3f}")

## 5) Learning curves across both stages

Concatenate stage 1 and stage 2 histories so the boundary is visible. Watch for the small lift right at the seam, that's the fine-tuning earning its keep.

In [ ]:
acc      = history_stage1.history["accuracy"]     + history_stage2.history["accuracy"]
val_acc  = history_stage1.history["val_accuracy"] + history_stage2.history["val_accuracy"]
boundary = len(history_stage1.history["accuracy"])

plt.figure(figsize=(10, 4))
plt.plot(acc, label="train")
plt.plot(val_acc, label="val")
plt.axvline(boundary - 0.5, color="k", linestyle="--", alpha=0.5,
            label="unfreeze + drop lr to 1e-5")
plt.title("Transfer learning, both stages")
plt.xlabel("epoch"); plt.ylabel("accuracy")
plt.ylim(0.5, 1.0); plt.legend()
plt.tight_layout()
plt.show()

## 6) Final comparison: same test set, four approaches

Three notebooks, four models, one test set. The progression should be monotonic, each new approach beats the previous one.

In [ ]:
rows = [
    ("random chance",                                   1 / num_classes),
    ("classical (RF on 34 hand features, 06_01)",        0.785),
    ("small CNN from scratch (06_02)",                   0.86),    # typical
    ("transfer, frozen backbone (stage 1)",              stage1_acc),
    ("transfer, fine-tuned (stage 2)",                   stage2_acc),
]

print(f"{'model':<48s} {'test acc':>10s}")
print("-" * 60)
for name, acc in rows:
    print(f"{name:<48s} {acc:>10.3f}")

## 7) Where errors live now

Even at 95%+ accuracy, the same handful of class pairs persist as the hardest, the various crop classes blur into each other; built-up classes (Highway / Industrial / Residential) confuse each other. The model is now near the ceiling of what 64x64 satellite imagery makes inherently distinguishable.

In [ ]:
y_pred = tl_model.predict(ds_test, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(label_names, rotation=90, fontsize=9)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(label_names, fontsize=9)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Fine-tuned MobileNetV2 confusion matrix")
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(classification_report(y_te, y_pred, target_names=label_names))

## What we built (and where Module 6 ends up)

Across three notebooks on the same EuroSAT test set:

| Notebook | Approach | Human effort | Compute | Test accuracy |
| --- | --- | --- | --- | --- |
| [06_01](06_01_main_classical.ipynb) | Hand features + RF | high (feature engineering) | low | ~0.79 |
| [06_02](06_02_main_cnn.ipynb) | CNN from scratch (with augmentation) | low | medium | ~0.85 |
| **06_03** | Pretrained + frozen | very low | low | ~0.92 |
| **06_03** | Pretrained + fine-tuned | very low | medium | ~0.95+ |

Three real takeaways:

1. **Transfer learning is the default starting point in production.** Unless you have a *lot* of data and a *very* unusual domain, starting from a pretrained backbone wins on accuracy, training time, and human effort simultaneously. The from-scratch CNN in 06_02 is mostly a teaching tool, on a real project you'd skip it and go straight here.
2. **The lr drop in stage 2 matters more than the architecture choice.** Most of the failures of fine-tuning in the wild are 'used the same learning rate as stage 1 and forgot the pretrained features'. Drop the lr by 50-100x.
3. **There's a ceiling.** Even a fine-tuned ImageNet backbone tops out around 95% on EuroSAT-RGB at 64x64. The remaining errors are mostly genuine ambiguity in the imagery, more compute won't fix them. Bigger / multispectral / higher-resolution data would.

## Knobs to play with

- **Backbone choice**, swap `MobileNetV2` for `EfficientNetB0` (a few % accuracy, more parameters) or `ResNet50` (older, larger).
- **`FREEZE_UPTO`**, unfreeze fewer layers (more conservative, slower convergence) or more (more capacity to specialize, more risk of catastrophic forgetting).
- **Image size**, MobileNetV2 supports up to 224x224. Larger inputs see more detail and usually help, at proportionally more compute.
- **Augmentation strength**, `RandomRotation(0.2)` or add `RandomZoom(0.1)` for more aggressive regularization.
- **`N_SAMPLES`**, if you have the patience, bump this to 27000 (the full dataset). You'll watch transfer learning go from 95% to 97-98%.